# Iris Prediction

Iris Prediction Network

Main Sources:
 
  [Kaggle Dataset](https://www.kaggle.com/datasets/arshid/iris-flower-dataset/data), 
  [Pytorch Basic Layout](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html), 
  [Tensor Board for Visualising](https://docs.pytorch.org/tutorials/intermediate/tensorboard_tutorial.html)

## Setup: importing Packages + checking for a GPU

In [45]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.tensorboard import SummaryWriter

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using: {device}")

Using: mps


## Defining the neural network
Main things I changed from the reference:
- Linear : 4 &rarr; 10 &rarr; 5 &rarr; 3 (Random numbers, not much thought beyond that)
- ReLU &rarr; SiLU (I read somewhere that its better, more smooth)

In [46]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_silu_stack = nn.Sequential(
            nn.Linear(4, 10),
            nn.SiLU(),
            nn.Linear(10, 5),
            nn.SiLU(),
            nn.Linear(5, 3)
        )

    def forward(self, x):
        logits = self.linear_silu_stack(x)
        return logits
    
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (linear_silu_stack): Sequential(
    (0): Linear(in_features=4, out_features=10, bias=True)
    (1): SiLU()
    (2): Linear(in_features=10, out_features=5, bias=True)
    (3): SiLU()
    (4): Linear(in_features=5, out_features=3, bias=True)
  )
)


## Importing Data, changing variety to numbers, splitting into traindata and testdata

In [47]:
data_path = './Data/IRIS.csv'

df = pd.read_csv(data_path)

df["species"] = df["species"].map({
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2,
})

df_tensor = torch.from_numpy(df.values).float()
X = df_tensor[:, :-1]
y = df_tensor[:, -1].long()

dataset = torch.utils.data.TensorDataset(X, y)
traindata, testdata = torch.utils.data.random_split(dataset, [0.80, 0.20])

print(X.size(), y.size(), len(traindata), len(testdata))

torch.Size([150, 4]) torch.Size([150]) 120 30


## Batching

In [48]:
batch_size = 10 # Chose a small batch because the dataset is small
train_loader = torch.utils.data.DataLoader(traindata, batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(testdata, batch_size, shuffle=False)

## Learning
- Changed the optimisation model from SGD to AdamW (cause I read its better)

In [49]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001) # learning rate adjusted because of the small dataset

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # if batch % 3 == 0:
        #     loss, current = loss.item(), (batch + 1) * len(X)
        #     print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

epochs = 100
for t in range(epochs):
    print(f"Epoch {t+1}:")
    train(train_loader, model, loss_fn, optimizer)
    test(test_loader, model, loss_fn)

print("Done!")

torch.save(model.state_dict(), "./Data/model.pth")
print("Saved PyTorch Model State to model.pth")

Epoch 1:
Accuracy: 30.0%, Avg loss: 1.095196 

Epoch 2:
Accuracy: 30.0%, Avg loss: 1.088096 

Epoch 3:
Accuracy: 30.0%, Avg loss: 1.080041 

Epoch 4:
Accuracy: 30.0%, Avg loss: 1.072151 

Epoch 5:
Accuracy: 30.0%, Avg loss: 1.059595 

Epoch 6:
Accuracy: 30.0%, Avg loss: 1.043768 

Epoch 7:
Accuracy: 30.0%, Avg loss: 1.025017 

Epoch 8:
Accuracy: 30.0%, Avg loss: 1.004786 

Epoch 9:
Accuracy: 30.0%, Avg loss: 0.979861 

Epoch 10:
Accuracy: 30.0%, Avg loss: 0.950383 

Epoch 11:
Accuracy: 30.0%, Avg loss: 0.928480 

Epoch 12:
Accuracy: 30.0%, Avg loss: 0.898810 

Epoch 13:
Accuracy: 30.0%, Avg loss: 0.873664 

Epoch 14:
Accuracy: 33.3%, Avg loss: 0.844463 

Epoch 15:
Accuracy: 33.3%, Avg loss: 0.825796 

Epoch 16:
Accuracy: 36.7%, Avg loss: 0.798236 

Epoch 17:
Accuracy: 40.0%, Avg loss: 0.773070 

Epoch 18:
Accuracy: 46.7%, Avg loss: 0.747395 

Epoch 19:
Accuracy: 46.7%, Avg loss: 0.729246 

Epoch 20:
Accuracy: 46.7%, Avg loss: 0.710671 

Epoch 21:
Accuracy: 50.0%, Avg loss: 0.680181 

E

## Evaluating the model using softmax + argmax prediction of the most likely variation (just a little bit more visual)

In [50]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("./Data/model.pth", weights_only=True))

classes = [
    "Iris-setosa",
    "Iris-versicolor",
    "Iris-virginica"
]

# Rotate throuh i through basically the whole test set

for i in range(len(testdata)):
    total = 0.0
    random_row = np.random.randint(0, len(testdata))
    example = testdata[random_row][0].unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad(): # both of these turn off processes that are only needed for training
        prediction = model(example)
        probabilities = torch.softmax(prediction, dim=1)
        likely_variety = classes[probabilities.argmax()]
        total = total + max(probabilities.flatten().cpu().numpy())
        print(f"Predicted variety: {likely_variety}, Probabilities: {probabilities.cpu().numpy()}")
        print(f"Percentage \"correct\": {total/(1):.2%}")
        i += 1

Predicted variety: Iris-virginica, Probabilities: [[3.9100889e-13 1.0689117e-02 9.8931086e-01]]
Percentage "correct": 98.93%
Predicted variety: Iris-virginica, Probabilities: [[7.0253670e-10 9.6757069e-02 9.0324295e-01]]
Percentage "correct": 90.32%
Predicted variety: Iris-virginica, Probabilities: [[2.3668895e-11 2.0401340e-02 9.7959864e-01]]
Percentage "correct": 97.96%
Predicted variety: Iris-versicolor, Probabilities: [[0.00127834 0.9542507  0.04447101]]
Percentage "correct": 95.43%
Predicted variety: Iris-virginica, Probabilities: [[5.8260586e-15 1.2388033e-03 9.9876118e-01]]
Percentage "correct": 99.88%
Predicted variety: Iris-versicolor, Probabilities: [[1.15587405e-04 9.72601354e-01 2.72830222e-02]]
Percentage "correct": 97.26%
Predicted variety: Iris-versicolor, Probabilities: [[1.15587405e-04 9.72601354e-01 2.72830222e-02]]
Percentage "correct": 97.26%
Predicted variety: Iris-virginica, Probabilities: [[5.4457779e-11 5.4165743e-02 9.4583428e-01]]
Percentage "correct": 94.58%


# Summary
This may be overfitted. I am getting results around 99% most of the time using the weights.
Things I could still improve:
- Stratify the split
- Benchmark the model using different variables
- Visualise the training better using e.g. sns or TensorBoard
- Test the model further using e.g. a larger dataset

I also wanted to present it in a much fancier way but using a Jupyter Notebook ended up being a lot simpler.